In [6]:
# ============ ABLATION STUDY: Cleaning components ============
import re, json, os
# --- Path fix: notebook'un nereden çalıştırıldığına göre cache yolunu bul ---
import os
def find_cache_dir():
    for c in ['baseline/cache', 'cache', '../baseline/cache']:
        if os.path.exists(c):
            return c
    raise FileNotFoundError("Cannot find baseline/cache directory")

CACHE_DIR = find_cache_dir()
RESULTS_DIR = os.path.join(os.path.dirname(CACHE_DIR), 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Cache: {CACHE_DIR}")
print(f"Results: {RESULTS_DIR}")
from datetime import datetime
from tqdm import tqdm
import torch
import numpy as np
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset, Dataset



# Setup
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
batch_size = 16
lang = "fr"
print(f"Device: {device}")

# Cache + model
collection_embeddings = torch.load(os.path.join(CACHE_DIR, 'collection_embeddings.pt'))
collection_keys = np.load(os.path.join(CACHE_DIR, 'collection_keys.npy'))
e5_name = 'intfloat/multilingual-e5-large'
e5_tokenizer = AutoTokenizer.from_pretrained(e5_name)
e5_model = AutoModel.from_pretrained(e5_name).to(device).eval()
print("E5 loaded.")

def average_pool(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

def embed_texts(texts, max_len=128, desc=""):
    ds = Dataset.from_dict({"text": [f'query: {t}' for t in texts]})
    def tok(ex):
        return e5_tokenizer(ex["text"], max_length=max_len, truncation=True, padding="max_length", return_tensors="pt")
    ds = ds.map(tok, batched=True, remove_columns=ds.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
    embs = []
    with torch.inference_mode():
        for batch in DataLoader(ds, batch_size=batch_size, num_workers=0):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = e5_model(**batch)
            embs.append(average_pool(outputs.last_hidden_state, batch['attention_mask']).detach().cpu())
    return F.normalize(torch.vstack(embs), p=2, dim=1)

def mrr_at_5(cos_sims, labels):
    top5_idxs = np.argsort(-cos_sims, axis=1)[:, :5]
    top5_keys = collection_keys[top5_idxs]
    labels_2d = labels[:, None]
    matches_mask = (top5_keys == labels_2d)
    positions = np.argmax(matches_mask, axis=1)
    positions[~matches_mask.any(axis=1)] = -1
    rr = np.zeros_like(positions, dtype=float)
    mask = positions >= 0
    rr[mask] = 1.0 / (positions[mask] + 1)
    return float(rr.mean())

# Veriyi yükle (FR dev)
data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", lang)
dev_raw = data["dev"]
dev_labels = np.array(dev_raw["pubkey"])
dev_raw_texts = list(dev_raw["text"])
print(f"FR Dev: {len(dev_raw_texts)} tweets")

# Ablation variants
def remove_url(t):       return re.sub(r'http\S+|www\.\S+', '', t)
def remove_mention(t):   return re.sub(r'@\w+', '', t)
def remove_hashtag(t):   return re.sub(r'#(\w+)', r'\1', t)
def remove_numbering(t): 
    t = re.sub(r'^\d+/n\s*Fact\s*\d+\s*:\s*', '', t, flags=re.IGNORECASE)
    t = re.sub(r'^\d+/\d+\s*', '', t)
    return t
def remove_emoji(t):     return re.sub(r'[^\w\s\.\,\:\;\!\?\-\(\)\']+', ' ', t)
def collapse_ws(t):      return re.sub(r'\s+', ' ', t).strip()

def full_clean(t):
    t = remove_url(t)
    t = remove_mention(t)
    t = remove_hashtag(t)
    t = remove_numbering(t)
    t = remove_emoji(t)
    return collapse_ws(t)

variants = {
    "baseline (no clean)": lambda t: t,
    "only URL removal":    lambda t: collapse_ws(remove_url(t)),
    "only @mention":       lambda t: collapse_ws(remove_mention(t)),
    "only #hashtag":       lambda t: collapse_ws(remove_hashtag(t)),
    "only numbering":      lambda t: collapse_ws(remove_numbering(t)),
    "only emoji/special":  lambda t: collapse_ws(remove_emoji(t)),
    "ALL combined":        full_clean,
}

# Run each variant
print("\n" + "=" * 60)
print(f"{'Variant':30s}  MRR@5     Δ vs baseline")
print("=" * 60)

results = {}
baseline_score = None

for name, fn in tqdm(variants.items(), desc="Variants"):
    cleaned = [fn(t) for t in dev_raw_texts]
    emb = embed_texts(cleaned, desc=name)
    score = mrr_at_5((emb @ collection_embeddings.T).numpy(), dev_labels)
    results[name] = score
    if baseline_score is None:
        baseline_score = score
    delta = score - baseline_score
    print(f"{name:30s}  {score:.4f}   {delta:+.4f}")

# Save
ts = datetime.now().isoformat()
for name, score in results.items():
    log = {
        "timestamp": ts,
        "experiment": "exp7_ablation_fr_dev",
        "variant": name,
        "language": "fr",
        "mrr5_dev": score,
        "delta_vs_baseline": score - baseline_score,
    }
    with open(os.path.join(RESULTS_DIR, 'results.jsonl'), 'a') as f:
        f.write(json.dumps(log) + '\n')

print("\nLogged.")


Cache: cache
Results: results
Device: mps


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7702.30it/s]


E5 loaded.
FR Dev: 702 tweets

Variant                         MRR@5     Δ vs baseline


Variants:  14%|█▍        | 1/7 [00:34<03:24, 34.05s/it]

baseline (no clean)             0.4584   +0.0000


Variants:  29%|██▊       | 2/7 [01:05<02:42, 32.45s/it]

... (7 outputs left out)

Variants:  86%|████████▌ | 6/7 [03:12<00:32, 32.21s/it]

only emoji/special              0.4627   +0.0043


Variants: 100%|██████████| 7/7 [03:47<00:00, 32.46s/it]

ALL combined                    0.4620   +0.0036

Logged.


In [7]:
# Optimum cleaning subset: numbering hariç hepsi
def optimal_clean(t):
    t = remove_url(t)
    t = remove_mention(t)
    t = remove_hashtag(t)
    t = remove_emoji(t)
    return collapse_ws(t)

cleaned = [optimal_clean(t) for t in dev_raw_texts]
emb = embed_texts(cleaned)
score = mrr_at_5((emb @ collection_embeddings.T).numpy(), dev_labels)
print(f"\nOptimal (no numbering): {score:.4f}  (Δ vs ALL = {score - 0.4620:+.4f})")

# Aynı şekilde log
log = {
    "timestamp": datetime.now().isoformat(),
    "experiment": "exp7_ablation_optimal_fr_dev",
    "variant": "all_except_numbering",
    "language": "fr",
    "mrr5_dev": score,
    "delta_vs_all_combined": score - 0.4620,
}
with open(os.path.join(RESULTS_DIR, 'results.jsonl'), 'a') as f:
    f.write(json.dumps(log) + '\n')

Map: 100%|██████████| 702/702 [00:00<00:00, 7636.09 examples/s]



Optimal (no numbering): 0.4613  (Δ vs ALL = -0.0007)
